In [2]:
import pandas as pd
import numpy as np

# Load the selection saved from EDA
df_model = pd.read_csv('../data/loan_clean_eda.csv')

df_clean = df_model.copy()

# FIX: use assignment instead of inplace=True
# inplace=True on a copy raises SettingWithCopyWarning in Pandas 2.x
num_cols_with_missing = ['mort_acc', 'revol_util', 'pub_rec_bankruptcies']

for col in num_cols_with_missing:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)  # no inplace!
    print(f'{col}: filled with median = {median_val:.2f}')

# Verify — all should be 0
print('\nMissing values remaining:')
print(df_clean.isnull().sum())

mort_acc: filled with median = 2.00
revol_util: filled with median = 50.47
pub_rec_bankruptcies: filled with median = 0.00

Missing values remaining:
loan_amnt               0
int_rate                0
installment             0
annual_inc              0
dti                     0
delinq_2yrs             0
inq_last_6mths          0
open_acc                0
pub_rec                 0
revol_bal               0
revol_util              0
total_acc               0
mort_acc                0
pub_rec_bankruptcies    0
default                 0
dtype: int64


In [3]:
# 1. Total DTI — adds new-loan burden to existing debt ratio
# FIX: use 12.0 for explicit float division
df_clean['total_dti'] = (df_clean['dti'] +
(df_clean['installment'] / (df_clean['annual_inc'] / 12.0 + 1)) * 100)
# 2. Loan-to-Income ratio (how big is this loan vs their income?)
df_clean['loan_income_ratio'] = df_clean['loan_amnt'] / (df_clean['annual_inc'] + 1)
# 3. Credit utilization flag (high util = financial stress)
df_clean['high_utilization'] = (df_clean['revol_util'] > 75).astype(int)
# 4. Has any derogatory record
df_clean['has_derogatory'] = (
(df_clean['pub_rec'] > 0) |
(df_clean['pub_rec_bankruptcies'] > 0)
).astype(int)
# 5. Inquiry risk flag (3+ inquiries = desperately seeking credit)
df_clean['high_inquiries'] = (df_clean['inq_last_6mths'] >= 3).astype(int)
# 6. Monthly payment burden
df_clean['monthly_income'] = df_clean['annual_inc'] / 12.0
df_clean['payment_to_income']= df_clean['installment'] / (df_clean['monthly_income'] + 1)
print('Engineered features created successfully')
print('Dataset shape now:', df_clean.shape)

Engineered features created successfully
Dataset shape now: (50000, 22)


In [4]:
# Cap income at 99th percentile (remove billionaires / data errors)
income_cap = df_clean['annual_inc'].quantile(0.99)
print(f'Capping annual_inc above: ${income_cap:,.0f}')
# FIX: .loc[] + reset_index to avoid index misalignment
df_clean = df_clean.loc[df_clean['annual_inc'] <= income_cap].copy()
df_clean = df_clean.reset_index(drop=True)
# Cap loan_amnt at 99th percentile (clip, not filter)
loan_cap = df_clean['loan_amnt'].quantile(0.99)
df_clean['loan_amnt'] = df_clean['loan_amnt'].clip(upper=loan_cap)
print(f'Final dataset shape: {df_clean.shape}')
print(f'Default rate: {df_clean["default"].mean()*100:.2f}%')


Capping annual_inc above: $198,115
Final dataset shape: (49500, 22)
Default rate: 40.10%


In [5]:
FEATURES = [
 "loan_amnt", "int_rate", "installment", "annual_inc", "dti",
 "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec",
 "revol_bal", "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies",
 # Engineered features
 "total_dti", "loan_income_ratio", "high_utilization",
 "has_derogatory", "high_inquiries", "payment_to_income"
]
TARGET = "default"
X = df_clean[FEATURES]
y = df_clean[TARGET]
print(f"Features: {len(FEATURES)}")
print(f"Samples: {len(X)}")
print(f"Default rate: {y.mean()*100:.2f}%")
df_clean[FEATURES + [TARGET]].to_csv("../data/loan_clean.csv", index=False)
print("Saved to data/loan_clean.csv")

Features: 20
Samples: 49500
Default rate: 40.10%
Saved to data/loan_clean.csv
